In [ ]:
from discovery_utils.synthesis.policy import policy_update
from discovery_utils.utils import google
import pandas as pd

In [ ]:
sheet_id = "1w2nSas1LwmPQY9HK-drrxIdGpDV3wU3pePDibHPVqL8"

In [ ]:
HansardData = policy_update.HansardData()

In [ ]:
missions = ["ASF"]

In [ ]:
_, data_signals = policy_update.create_policy_update_message(
    Hansard=HansardData,
    missions=missions,
    message_date="2025-02-19",
    data_start_date="2025-01-01",
    data_end_date="2025-03-31",
)

In [ ]:
debates_cols = [
    "date",
    "title",
    "purpose",
    "positives",
    "negatives",
    "next_steps",
]

In [ ]:
debates_df = (
    pd.DataFrame(data_signals[0][0]['signal'])
    .sort_values("date", ascending=True)
    .assign(
        positives = lambda df: df.positives.apply(lambda x: "\n".join(x)),
        negatives = lambda df: df.negatives.apply(lambda x: "\n".join(x)),
        next_steps = lambda df: df.next_steps.apply(lambda x: "\n".join(x)),
    )
)[debates_cols]

In [ ]:
google.upload_data_to_gsheet(sheet_id, {"commons_debates": debates_df})
google.format_gsheet(sheet_id, "commons_debates", freeze_cols=2)

In [ ]:
cols_highlights = [
    "date",
    "heading",
    "summary",
    "url",
    "mission_labels",
    "topic_labels",
]

In [ ]:
highlights_df = []
for _debate in data_signals[0][1]['signal']:
    highlights_df.append((
        pd.DataFrame(_debate['quotes'])
        .assign(
            heading = _debate['heading'],
            date = _debate['date'],
        )
    ))
highlights_df = (
    pd.concat(highlights_df, ignore_index=True)
    .sort_values(["date", "heading"], ascending=True)
    .assign(speech_id = lambda df: df.url.apply(lambda x: "uk.org.publicwhip/debate/" + x.split("?id=")[-1]))
    .merge(
        HansardData.labelstore_df[["id", "mission_labels", "topic_labels"]],
        left_on="speech_id",
        right_on="id",
        how="left",
    )
)[cols_highlights]

In [ ]:
highlights_df

In [ ]:
google.upload_data_to_gsheet(sheet_id, {"commons_highlights": highlights_df})
google.format_gsheet(sheet_id, "commons_highlights", freeze_cols=0)

## Keyword counts

In [ ]:
import datetime
def get_quarter_from_date(date:str) -> int:
    """Return the quarter number from a given YYYY-MM-DD date string."""
    _date = datetime.datetime.strptime(date, "%Y-%m-%d")
    return (_date.month-1)//3 + 1

In [ ]:
speeches_df = (
    HansardData.debates_df
    .query("date >= '2020-01-01' & date <= '2025-03-31'")
    .merge(
        HansardData.labelstore_df[['id', 'mission_labels', 'topic_labels']],
        left_on='speech_id',
        right_on='id',
        how='left'
    )
    .assign(mission_labels = lambda df: df.mission_labels.apply(lambda x: x.split(",") if (type(x) is str) else []))
    .assign(topic_labels = lambda df: df.topic_labels.apply(lambda x: x.split(",") if (type(x) is str) else []))    
    .explode("mission_labels")
    .query("mission_labels in @missions")
    .explode("topic_labels")
    .assign(quarter = lambda df: df.date.apply(get_quarter_from_date))
    .assign(quarter = lambda df: df.year.astype(str) + "-Q" + df.quarter.astype(str))
)

In [ ]:
ts_hp = (
    speeches_df
    .query("topic_labels == 'Heat pumps'")
    .groupby("quarter")
    .agg({"speech_id": "count"})
    .reset_index()
)

In [ ]:
from discovery_utils.utils import charts

In [ ]:
ts_hp

In [ ]:
charts.ts_bar(
    ts_hp,
    variable="speech_id",
    variable_title="Number of speeches",
    time_column="quarter",

)


In [ ]:
HansardData.debates_df.head(1)

In [ ]:
HansardData.labelstore_df.tail(10)